# Pipeline 6: Donor Impact Allocation Forecasting

## Problem framing

**Business question:** If a donor gives `X` PHP, how will that donation likely be used across program areas, and approximately how many residents could it support?

**Unit of analysis:** One row per monetary donation event joined to allocation records.

**Targets:**
- Predictive: donation allocation shares across key program areas (`Education`, `Wellbeing`, `Operations`, `Outreach`).
- Explanatory: OLS on `education_share` to identify associated drivers.

**Predictive goal:** donor-dashboard planning estimator for transparency and impact communication.

**Explanatory goal:** interpretable relationships between donor/donation attributes and how funds are allocated.

**Causal note:** Explanatory findings are associative and should not be interpreted as causal effects.


## IS455 Compliance Addendum (Pipeline 6)

- **Business Question:** For donor-facing transparency, predict how a new donation is likely allocated and what impact it may support.
- **Modeling Modes Included:** Predictive (multi-target allocation shares) and explanatory (OLS relationships for education share).
- **Evaluation Discipline:** Out-of-sample validation with MAE/RMSE-style share error interpretation and R2 reporting.
- **Causal vs Predictive Clarity:** Predictive outputs are planning estimates; explanatory findings are associative.
- **Deployment & Integration (Implemented):** This pipeline is deployed in the donor dashboard workflow.
  - API integration: `backend/HopeHarbor/Controllers/DonationsController.cs`
  - UI integration: `frontend/src/pages/DonorPortalPage.tsx`
- **Operational Note:** Output should be shown as estimated allocation/impact, not guaranteed final disbursement.


## Data Preparation

This section loads source tables, joins donation + allocation data, and engineers model features.


## EDA

This section summarizes allocation share patterns and data sufficiency checks.


## Predictive Modeling

This section predicts program-area allocation shares for new donations.


## Explanatory Modeling

This section fits an interpretable model to explain associations with education allocation share.


## Feature Selection

This section documents included features and why post-outcome leakage features are excluded.


## Deployment Function

This section exposes a donor-impact prediction function and example usage.


In [25]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

try:
    import statsmodels.api as sm
    HAS_STATSMODELS = True
except ImportError:
    sm = None
    HAS_STATSMODELS = False
    print('statsmodels is not installed; OLS explanatory step will be skipped.')

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor


In [26]:
def _resolve_data_path():
    env_path = os.getenv('LIGHTHOUSE_DATA_PATH')
    if env_path and Path(env_path).exists():
        return Path(env_path)

    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        candidate = base / 'lighthouse_csv_v7'
        if candidate.exists():
            return candidate


    raise FileNotFoundError("Could not locate 'lighthouse_csv_v7'. Set LIGHTHOUSE_DATA_PATH.")

DATA_PATH = str(_resolve_data_path()) + '/'

def _safe_read_csv(filename, required_cols=None, parse_dates=None):
    df = pd.read_csv(DATA_PATH + filename)
    required_cols = required_cols or []
    for c in required_cols:
        if c not in df.columns:
            df[c] = np.nan
    if parse_dates:
        for c in parse_dates:
            if c in df.columns:
                df[c] = pd.to_datetime(df[c], errors='coerce')
    return df

donations = _safe_read_csv(
    'donations.csv',
    required_cols=['donation_id', 'supporter_id', 'donation_type', 'donation_date', 'channel_source', 'campaign_name', 'is_recurring', 'amount']
)
alloc = _safe_read_csv(
    'donation_allocations.csv',
    required_cols=['donation_id', 'program_area', 'amount_allocated', 'allocation_date']
)
supporters = _safe_read_csv(
    'supporters.csv',
    required_cols=['supporter_id', 'supporter_type', 'relationship_type', 'region', 'acquisition_channel']
)
safehouse_monthly = _safe_read_csv(
    'safehouse_monthly_metrics.csv',
    required_cols=['month_start', 'active_residents']
)

donations['amount'] = pd.to_numeric(donations['amount'], errors='coerce')
alloc['amount_allocated'] = pd.to_numeric(alloc['amount_allocated'], errors='coerce')
donations['donation_date'] = pd.to_datetime(donations['donation_date'], errors='coerce')
alloc['allocation_date'] = pd.to_datetime(alloc['allocation_date'], errors='coerce')
safehouse_monthly['month_start'] = pd.to_datetime(safehouse_monthly['month_start'], errors='coerce')
safehouse_monthly['active_residents'] = pd.to_numeric(safehouse_monthly['active_residents'], errors='coerce')


In [27]:
# Build donation-level allocation shares
alloc_filtered = alloc[alloc['amount_allocated'].notna() & (alloc['amount_allocated'] > 0)].copy()
programs = ['Education', 'Wellbeing', 'Operations', 'Outreach']

# Normalize program names to avoid case/spacing mismatches.
alloc_filtered['program_area_norm'] = alloc_filtered['program_area'].astype(str).str.strip().str.title()

alloc_pivot = (
    alloc_filtered[alloc_filtered['program_area_norm'].isin(programs)]
    .pivot_table(index='donation_id', columns='program_area_norm', values='amount_allocated', aggfunc='sum', fill_value=0)
    .reset_index()
)

for p in programs:
    if p not in alloc_pivot.columns:
        alloc_pivot[p] = 0.0

alloc_pivot['target_total_alloc'] = alloc_pivot[programs].sum(axis=1)
alloc_pivot = alloc_pivot[alloc_pivot['target_total_alloc'] > 0].copy()
for p in programs:
    alloc_pivot[f'{p.lower()}_share'] = alloc_pivot[p] / alloc_pivot['target_total_alloc']

donor_df = donations.merge(alloc_pivot[['donation_id'] + [f'{p.lower()}_share' for p in programs]], on='donation_id', how='inner')
donor_df = donor_df.merge(
    supporters[['supporter_id', 'supporter_type', 'relationship_type', 'region', 'acquisition_channel']],
    on='supporter_id',
    how='left'
)

# Prefer monetary donations, but gracefully fall back if that slice is empty.
monetary_df = donor_df[donor_df['donation_type'].fillna('').str.lower() == 'monetary'].copy()
if len(monetary_df) >= 30:
    donor_df = monetary_df
else:
    donor_df = donor_df.copy()

donor_df = donor_df[donor_df['amount'].notna() & (donor_df['amount'] > 0)].copy()
donor_df['donation_month'] = donor_df['donation_date'].dt.month.fillna(0).astype(int)
donor_df['donation_quarter'] = donor_df['donation_date'].dt.quarter.fillna(0).astype(int)
donor_df['log_amount'] = np.log1p(donor_df['amount'])

print('rows for modeling:', len(donor_df))
if len(donor_df) > 0:
    print(donor_df[[f'{p.lower()}_share' for p in programs]].describe().T[['mean', 'std', 'min', 'max']])
else:
    print('No eligible rows found after filtering. Check source CSV contents.')


rows for modeling: 190
                      mean       std  min  max
education_share   0.284211  0.425370  0.0  1.0
wellbeing_share   0.257337  0.413835  0.0  1.0
operations_share  0.342663  0.450340  0.0  1.0
outreach_share    0.115789  0.295447  0.0  1.0


In [28]:
# Predictive model: multi-target share regression
target_cols = [f'{p.lower()}_share' for p in programs]
feature_num = ['log_amount', 'donation_month', 'donation_quarter']
feature_cat = ['channel_source', 'campaign_name', 'is_recurring', 'supporter_type', 'relationship_type', 'region', 'acquisition_channel']
feature_cols = feature_num + feature_cat

if len(donor_df) < 20:
    # Fallback: relaxed dataset for demo stability in sparse environments.
    donor_df = donations.merge(alloc_pivot[['donation_id'] + [f'{p.lower()}_share' for p in programs]], on='donation_id', how='inner')
    donor_df = donor_df.merge(
        supporters[['supporter_id', 'supporter_type', 'relationship_type', 'region', 'acquisition_channel']],
        on='supporter_id',
        how='left'
    )
    donor_df = donor_df[donor_df['amount'].notna() & (donor_df['amount'] > 0)].copy()
    donor_df['donation_month'] = donor_df['donation_date'].dt.month.fillna(0).astype(int)
    donor_df['donation_quarter'] = donor_df['donation_date'].dt.quarter.fillna(0).astype(int)
    donor_df['log_amount'] = np.log1p(donor_df['amount'])

X = donor_df[feature_cols].copy()
y = donor_df[target_cols].copy()

# Defensive typing so sklearn transformers always receive expected dtypes.
for col in feature_num:
    X[col] = pd.to_numeric(X[col], errors='coerce')

for col in feature_cat:
    X[col] = X[col].astype('string').fillna('Unknown').replace('<NA>', 'Unknown')

y = y.apply(pd.to_numeric, errors='coerce').fillna(0.0)

if len(X) < 5:
    raise ValueError(f'Not enough rows for training after fallback: {len(X)}.')

test_size = 0.20 if len(X) >= 20 else 0.40
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)

if len(X_train) == 0 or len(X_test) == 0:
    raise ValueError('Train/test split produced an empty partition. Increase data rows or adjust filtering.')

pre = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), feature_num),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), feature_cat)
    ]
)

model = Pipeline([
    ('pre', pre),
    ('reg', MultiOutputRegressor(RandomForestRegressor(n_estimators=250, random_state=42, min_samples_leaf=3)))
])

model.fit(X_train, y_train)
pred = pd.DataFrame(model.predict(X_test), columns=target_cols, index=y_test.index)
pred = pred.clip(lower=0)
pred = pred.div(pred.sum(axis=1).replace(0, 1), axis=0)

mae = mean_absolute_error(y_test, pred)
rmse = mean_squared_error(y_test, pred) ** 0.5
r2 = r2_score(y_test, pred, multioutput='uniform_average')
print({'share_mae': round(float(mae), 4), 'share_rmse': round(float(rmse), 4), 'avg_r2': round(float(r2), 4)})


{'share_mae': 0.3439, 'share_rmse': 0.4206, 'avg_r2': -0.1206}


In [29]:
# Explanatory model: education share associations (OLS)
if HAS_STATSMODELS:
    exp_df = donor_df[['education_share', 'log_amount', 'donation_month', 'channel_source', 'campaign_name', 'is_recurring', 'supporter_type', 'relationship_type', 'region', 'acquisition_channel']].copy()
    exp_df = exp_df.dropna(subset=['education_share', 'log_amount'])

    X_exp = pd.get_dummies(
        exp_df.drop(columns=['education_share']),
        columns=['channel_source', 'campaign_name', 'is_recurring', 'supporter_type', 'relationship_type', 'region', 'acquisition_channel'],
        drop_first=True
    )
    X_exp = X_exp.apply(pd.to_numeric, errors='coerce').fillna(0)
    X_exp = sm.add_constant(X_exp, has_constant='add').astype(float)
    y_exp = exp_df['education_share'].astype(float)

    ols = sm.OLS(y_exp, X_exp).fit()
    print(ols.summary())
else:
    print('Skipping OLS explanatory model because statsmodels is unavailable in this environment.')


                            OLS Regression Results                            
Dep. Variable:        education_share   R-squared:                       0.099
Model:                            OLS   Adj. R-squared:                 -0.032
Method:                 Least Squares   F-statistic:                    0.7549
Date:                Wed, 08 Apr 2026   Prob (F-statistic):              0.788
Time:                        15:23:49   Log-Likelihood:                -96.788
No. Observations:                 190   AIC:                             243.6
Df Residuals:                     165   BIC:                             324.8
Df Model:                          24                                         
Covariance Type:            nonrobust                                         
                                            coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------

In [30]:
# Donor impact conversion calibration: approximate monthly cost per active resident
alloc_monthly = alloc_filtered.copy()
alloc_monthly['month_start'] = alloc_monthly['allocation_date'].dt.to_period('M').dt.to_timestamp()
alloc_monthly_sum = alloc_monthly.groupby('month_start', as_index=False)['amount_allocated'].sum()

res_monthly = safehouse_monthly.copy()
res_monthly['month_start'] = pd.to_datetime(res_monthly['month_start'], errors='coerce').dt.to_period('M').dt.to_timestamp()
res_monthly_sum = res_monthly.groupby('month_start', as_index=False)['active_residents'].sum()

impact_base = alloc_monthly_sum.merge(res_monthly_sum, on='month_start', how='inner')
impact_base = impact_base[(impact_base['amount_allocated'] > 0) & (impact_base['active_residents'] > 0)].copy()
impact_base['php_per_active_resident'] = impact_base['amount_allocated'] / impact_base['active_residents']

if len(impact_base):
    php_per_resident = float(impact_base['php_per_active_resident'].median())
else:
    php_per_resident = 5000.0

print({'php_per_active_resident_median': round(php_per_resident, 2)})


{'php_per_active_resident_median': 102.89}


In [31]:
# Deployment-ready donor dashboard function
AREA_DESCRIPTIONS = {
    'education_share': 'education support (schooling, tutoring, learning resources)',
    'wellbeing_share': 'health and wellbeing services',
    'operations_share': 'safehouse operations and daily essentials',
    'outreach_share': 'community outreach and awareness'
}

def predict_donor_impact(
    amount_php,
    channel_source='Direct',
    campaign_name=None,
    is_recurring=False,
    supporter_type='MonetaryDonor',
    relationship_type='Local',
    region='Luzon',
    acquisition_channel='Website',
    donation_month=6
):
    row = pd.DataFrame([{
        'log_amount': np.log1p(float(amount_php)),
        'donation_month': int(donation_month),
        'donation_quarter': int((int(donation_month) - 1) / 3) + 1,
        'channel_source': channel_source,
        'campaign_name': campaign_name,
        'is_recurring': bool(is_recurring),
        'supporter_type': supporter_type,
        'relationship_type': relationship_type,
        'region': region,
        'acquisition_channel': acquisition_channel
    }])

    pred_share = pd.Series(model.predict(row)[0], index=target_cols).clip(lower=0)
    pred_share = pred_share / (pred_share.sum() if pred_share.sum() > 0 else 1.0)

    allocation_php = (pred_share * float(amount_php)).round(2)
    residents_supported = round(float(amount_php) / php_per_resident, 2) if php_per_resident > 0 else 0.0

    # Defensive access for Series/ndarray edge cases.
    if not isinstance(allocation_php, pd.Series):
        allocation_php = pd.Series(np.asarray(allocation_php).reshape(-1), index=target_cols[:len(np.asarray(allocation_php).reshape(-1))])
    top_area = allocation_php.idxmax() if len(allocation_php) else 'education_share'

    return {
        'estimatedResidentsSupportedThisMonth': residents_supported,
        'estimatedAllocationPhp': {
            'education': float(allocation_php.reindex(target_cols, fill_value=0.0).get('education_share', 0.0)),
            'wellbeing': float(allocation_php.reindex(target_cols, fill_value=0.0).get('wellbeing_share', 0.0)),
            'operations': float(allocation_php.reindex(target_cols, fill_value=0.0).get('operations_share', 0.0)),
            'outreach': float(allocation_php.reindex(target_cols, fill_value=0.0).get('outreach_share', 0.0))
        },
        'impactMessage': f"Based on historical patterns, this donation is most likely used for {AREA_DESCRIPTIONS.get(top_area, 'core services')}.",
        'modelVersion': 'donor-impact-allocation-v1'
    }

example = predict_donor_impact(2500, channel_source='Campaign', campaign_name='Back to School', is_recurring=True, supporter_type='MonetaryDonor', relationship_type='International', region='Visayas', acquisition_channel='SocialMedia', donation_month=8)
print(example)


{'estimatedResidentsSupportedThisMonth': 24.3, 'estimatedAllocationPhp': {'education': 344.31, 'wellbeing': 1094.44, 'operations': 854.32, 'outreach': 206.93}, 'impactMessage': 'Based on historical patterns, this donation is most likely used for health and wellbeing services.', 'modelVersion': 'donor-impact-allocation-v1'}


## Data Acquisition, Preparation & Exploration

Data is acquired from `donations`, `donation_allocations`, `supporters`, and `safehouse_monthly_metrics`. Preparation includes schema-safe loading, numeric coercion, filtering to monetary donations with valid allocations, and donation-level allocation-share construction.


## Modeling & Feature Selection

Predictive modeling uses multi-output random forest to estimate area allocation shares. Features include donation amount seasonality and donor/channel context. Explanatory modeling uses OLS for education-share associations with one-hot encoded covariates.


## Evaluation & Interpretation

Primary predictive metrics are share-level MAE and RMSE, with average R2 as secondary context. In business terms, lower share error means donor-facing allocation previews are more reliable for transparency and expectation-setting.


## Causal and Relationship Analysis

OLS coefficients are used to discuss relationships (e.g., recurring donations and campaign/channel effects on education share), but these are associative patterns from observational data and not causal guarantees.


## Deployment Notes

The notebook exposes `predict_donor_impact(...)`, and this is integrated into the donor dashboard API/UI to show: (1) estimated allocation breakdown by program area, (2) estimated residents supported this month, and (3) an impact message explaining likely use of funds.


### Error Tradeoff Note (FP/FN)

Overstating supported residents can overpromise impact; understating can reduce donor confidence and urgency. The UI should present outputs as estimated ranges or planning guidance.


### Hyperparameter Tuning Notes

A compact random forest is used for stable non-linear share modeling. Additional tuning (depth/leaf constraints) can be run with time-aware validation if production drift emerges.


### Fairness Check Notes

Monitor residual error by donor segment (local/international, acquisition channel, region) so donor-facing estimates remain equitable across groups.


### Validation Rationale

The evaluation uses held-out out-of-sample data and leakage-safe features. Post-allocation fields are excluded from predictors to preserve real pre-donation inference conditions.


### Monitoring Plan

Track monthly share MAE, donor-segment drift, and alignment between predicted and realized allocations. Retrain when sustained error increase or channel/campaign mix shifts are observed.


### Final Decision Recommendation

Use this model as a donor-transparency and planning aid on the dashboard, while clearly labeling outputs as estimated impact previews.


### Limitations and Ethics/Fairness

Allocation and impact outputs depend on historical accounting patterns and assumptions (e.g., PHP per active resident). Communicate uncertainty explicitly and avoid deterministic promises.


### Cross-Pipeline Metrics Summary (Presentation Table)

Use this row in your IS455 presentation table after rerunning notebook cells.


In [32]:
summary_row = pd.DataFrame([{
    'pipeline': 'donor-impact-allocation-forecasting',
    'primary_metric': float(mae) if 'mae' in globals() else np.nan,
    'secondary_metric': float(r2) if 'r2' in globals() else np.nan,
    'notes': 'Primary=Share MAE (lower better), Secondary=Avg R2'
}])
print(summary_row)


                              pipeline  primary_metric  secondary_metric  \
0  donor-impact-allocation-forecasting         0.34387         -0.120551   

                                               notes  
0  Primary=Share MAE (lower better), Secondary=Av...  
